In [1]:
import joblib
import numpy as np
import time
import os
import pandas as pd
from itertools import combinations
from scipy.stats import mode
import warnings
# 모든 FutureWarning(버전 변경 예정 경고) 무시
warnings.simplefilter(action='ignore', category=FutureWarning)

# Scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin

# 이미지 처리
from scipy.ndimage import center_of_mass, shift, zoom

In [2]:
# =========================================================
# 1. 헬퍼 클래스 정의
# =========================================================
# ==========================================
# 1. 사용자 정의 전처리 클래스
# ==========================================
class CustomImagePreprocess(BaseEstimator, TransformerMixin):
    def __init__(self, stroke_target=0.4, target_size=20, final_size=28, noise_threshold=0.02):
        self.stroke_target = stroke_target       
        self.target_size = target_size           
        self.final_size = final_size             
        self.noise_threshold = noise_threshold   
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        X = np.array(X)
        if X.ndim == 2:
            side = int(np.sqrt(X.shape[1]))
            X = X.reshape(-1, side, side)
        
        processed = []
        for img in X:
            img = self.normalize_scale(img)   
            img = self.remove_noise(img)      
            img = self.normalize_stroke(img)  
            img = self.normalize_size(img)    
            img = self.center_align(img)
            processed.append(img)
        
        processed = np.array(processed)
        return processed.reshape(len(processed), -1)
    
    # --- 내부 메서드 ---
    def normalize_scale(self, img):
        img = img.astype(np.float32)
        if img.max() > 1: img = img / 255.0
        return img
    
    def remove_noise(self, img):
        img = img.copy()
        img[img < self.noise_threshold] = 0.0
        return img
    
    def normalize_stroke(self, img):
        mask = img > 0.05
        if mask.sum() == 0: return img
        mean_val = img[mask].mean()
        if mean_val == 0: return img
        scale_factor = self.stroke_target / mean_val
        img = img * scale_factor
        img = np.clip(img, 0, 1)
        return img
    
    def normalize_size(self, img):
        coords = np.where(img > 0.05)
        if coords[0].size == 0: return np.zeros((self.final_size, self.final_size))
        r1, r2 = coords[0].min(), coords[0].max()
        c1, c2 = coords[1].min(), coords[1].max()
        digit = img[r1:r2+1, c1:c2+1]
        h, w = digit.shape
        if h == 0 or w == 0: return np.zeros((self.final_size, self.final_size))
        zoom_h = self.target_size / h
        zoom_w = self.target_size / w
        digit_resized = zoom(digit, (zoom_h, zoom_w))
        return digit_resized
    
    def center_align(self, img):
        canvas = np.zeros((self.final_size, self.final_size))
        h, w = img.shape
        if h > self.final_size or w > self.final_size:
            h_start = (h - self.final_size) // 2
            w_start = (w - self.final_size) // 2
            img = img[h_start:h_start+self.final_size, w_start:w_start+self.final_size]
            h, w = img.shape
        y0 = (self.final_size - h) // 2
        x0 = (self.final_size - w) // 2
        canvas[y0:y0+h, x0:x0+w] = img
        return canvas

class ForceFittedClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator):
        self.estimator = estimator
    def fit(self, X, y=None): return self
    def predict(self, X): return self.estimator.predict(X)
    def predict_proba(self, X): return self.estimator.predict_proba(X)
    def __sklearn_is_fitted__(self): return True

class PreFittedVotingClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimators, voting='soft'):
        self.estimators = estimators
        self.voting = voting
        
    def fit(self, X, y): return self
    
    def predict(self, X):
        if self.voting == 'soft':
            probas = self.predict_proba(X)
            return np.argmax(probas, axis=1)
        else: # hard voting
            predictions = np.zeros((X.shape[0], len(self.estimators)), dtype=int)
            for i, (_, model) in enumerate(self.estimators):
                predictions[:, i] = model.predict(X)
            try:
                majority_vote = mode(predictions, axis=1, keepdims=True).mode.flatten()
            except TypeError:
                majority_vote = mode(predictions, axis=1).mode.flatten()
            return majority_vote
    
    def predict_proba(self, X):
        if self.voting == 'hard':
            raise AttributeError("predict_proba is not available when voting='hard'")
        avg_proba = None
        count = 0
        for _, model in self.estimators:
            try:
                proba = model.predict_proba(X)
                if avg_proba is None: avg_proba = proba
                else: avg_proba += proba
                count += 1
            except: continue
        return avg_proba / count if count > 0 else avg_proba

In [3]:
# =========================================================
# 2. 데이터 로드
# =========================================================
def load_data_simple(hand_path, org_path):
    print("📂 Loading Data...")
    if not os.path.exists(hand_path) or not os.path.exists(org_path):
        return None, None, None, None
    d_hand = np.load(hand_path, allow_pickle=True)
    X_hand, y_hand = d_hand['x_train'], d_hand['y_train'].astype(int)
    d_org = np.load(org_path, allow_pickle=True)
    X_org, y_org = d_org['x_train'], d_org['y_train'].astype(int)
    
    if X_hand.ndim == 3: X_hand = X_hand.reshape(len(X_hand), -1)
    if X_org.ndim == 3: X_org = X_org.reshape(len(X_org), -1)
    X = np.concatenate([X_hand, X_org])
    y = np.concatenate([y_hand, y_org])
    if X.max() > 1.0: X = X.astype(np.float32) / 255.0
        
    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [4]:
# =========================================================
# 3. 메인 실행 코드
# =========================================================
if __name__ == "__main__":
    hand_file = '../data/final/mnist_final_balanced.npz'
    org_file = '../data/final/mnist_sklearn_40000.npz'
    X_train, X_test, y_train, y_test = load_data_simple(hand_file, org_file)

    if X_train is not None:
        # 1. Base 모델 5개 정의 (파일 경로 확인 필수!)
        model_paths = {
            'mlp': 'pipeline/base/mlp_HandAug_4x.joblib',
            'rf':  'pipeline/base/pipeline_model_HandAug_4x_Preprocessed_rf.pkl',
            'knn': 'pipeline/base/pipeline_knn_model_HandAug_4x_Preprocessed_KNN.pkl',
            'gb':  'pipeline/base/pipeline_model_HandAug_1x_Preprocessed_gb.pkl',
            # 'svm': 'pipeline/pipeline_svm_HandAug_4x_Preprocessed_SVM.joblib' # SVM 경로 확인 필요
        }
        
        # 2. 모델 로드 및 래핑
        loaded_estimators = {}
        print("\n📂 Loading Base Models...")
        for name, path in model_paths.items():
            if os.path.exists(path):
                try:
                    loaded_model = joblib.load(path)
                    wrapped_model = ForceFittedClassifier(loaded_model)
                    loaded_estimators[name] = wrapped_model
                    print(f"  > Loaded & Wrapped: {name}")
                except Exception as e:
                    print(f"  [Error] Failed to load {name}: {e}")
            else:
                print(f"  [Warning] File not found: {path} (Skipping {name})")
        
        available_models = list(loaded_estimators.items()) # [('mlp', model), ('rf', model), ...]
        
        # =========================================================================
        # 3. 메타 모델 설정 (6가지)
        # =========================================================================
        meta_configs = {
            'Logistic': {
                'model': LogisticRegression(random_state=42, max_iter=1000),
                'params': {'C': [0.1, 1.0, 10.0], 'solver': ['lbfgs', 'liblinear']}
            },
            'DecisionTree': {
                'model': DecisionTreeClassifier(random_state=42),
                'params': {'max_depth': [3, 4], 'min_samples_leaf': [10, 20]}
            },
            'KNN': {
                'model': KNeighborsClassifier(),
                'params': {'n_neighbors': [9, 15, 30], 'weights': ['uniform', 'distance']}
            }
        }

        if len(available_models) >= 3:
            results = []
            
            print("\n" + "="*60)
            print("🚀 Starting Combinations Experiment (Stacking 6 Meta & Voting)")
            print(f"   Base Models Available: {len(available_models)} ({[n for n, _ in available_models]})")
            print("="*60)

            # 4. 조합 생성 (3개 ~ 5개)
            min_k, max_k = 3, 5
            
            for k in range(min_k, min(max_k, len(available_models)) + 1):
                for combo in combinations(available_models, k):
                    # combo는 (('mlp', model), ('rf', model), ...) 형태의 튜플
                    combo_names = "+".join([name for name, _ in combo])
                    estimators_list = list(combo)
                    
                    print(f"\n[{k} Models Combo] {combo_names}")
                    
                    # -----------------------------------------------------
                    # (A) Stacking: Meta-Features 생성 (1회)
                    # -----------------------------------------------------
                    try:
                        # 임시 스태킹 모델로 transform 수행 -> Meta Features 추출
                        temp_stack = StackingClassifier(estimators=estimators_list, final_estimator=LogisticRegression(), cv='prefit', n_jobs=-1)
                        temp_stack.fit(X_train, y_train)
                        X_meta_train = temp_stack.transform(X_train)
                        
                        # (A-1) 6가지 Meta Model 실험
                        for meta_name, config in meta_configs.items():
                            # print(f"  -> Tuning Stacking Meta: {meta_name}...", end=" ")
                            
                            # GridSearchCV로 메타 모델 튜닝
                            grid = GridSearchCV(config['model'], config['params'], cv=3, scoring='accuracy', n_jobs=-1)
                            start = time.time()
                            grid.fit(X_meta_train, y_train)
                            elapsed = time.time() - start
                            
                            best_meta = grid.best_estimator_
                            
                            # 최종 Stacking 모델 생성
                            final_stack = StackingClassifier(
                                estimators=estimators_list,
                                final_estimator=best_meta,
                                cv='prefit',
                                n_jobs=-1
                            )
                            final_stack.fit(X_train, y_train)
                            acc = final_stack.score(X_test, y_test)
                            
                            print(f"Acc: {acc*100:.2f}% ({elapsed:.1f}s)")
                            
                            # 저장 및 기록
                            fname = f"stacking_{combo_names}_{meta_name}.joblib"
                            joblib.dump(final_stack, fname) # 필요시 주석 해제 (파일 너무 많음 주의)
                            
                            results.append({
                                'Type': 'Stacking',
                                'Base Models': combo_names,
                                'Meta Model': meta_name,
                                'Accuracy': acc,
                                'Time': elapsed
                            })

                    except Exception as e:
                        print(f"  [Error] Stacking failed for {combo_names}: {e}")

                    # -----------------------------------------------------
                    # (B) Voting (Soft & Hard)
                    # -----------------------------------------------------
                    for v_type in ['soft', 'hard']:
                        try:
                            # print(f"  -> Voting ({v_type})...", end=" ")
                            voting_clf = PreFittedVotingClassifier(estimators_list, voting=v_type)
                            voting_clf.fit(X_train, y_train)
                            
                            start = time.time()
                            acc_vote = accuracy_score(y_test, voting_clf.predict(X_test))
                            elapsed = time.time() - start
                            
                            print(f"Acc: {acc_vote*100:.2f}%")
                            
                            fname = f"voting_{v_type}_{combo_names}.joblib"
                            joblib.dump(voting_clf, fname)
                            
                            results.append({
                                'Type': f'Voting ({v_type})',
                                'Base Models': combo_names,
                                'Meta Model': '-',
                                'Accuracy': acc_vote,
                                'Time': elapsed
                            })
                        except Exception as e:
                            print(f"  [Error] Voting {v_type} failed: {e}")

            # 5. 결과 요약 및 저장
            print("\n" + "="*60)
            print("🏆 Top 20 Best Ensemble Results")
            print("="*60)
            df_res = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
            print(df_res.head(20))
            
            df_res.to_csv("ensemble_experiment_results.csv", index=False)
            print("\n  > Full results saved to 'ensemble_experiment_results.csv'")
            
            # (선택) 1등 모델 저장
            if len(df_res) > 0:
                best_row = df_res.iloc[0]
                print(f"\n🥇 Best Model: {best_row['Type']} | Base: {best_row['Base Models']} | Meta: {best_row['Meta Model']} ({best_row['Accuracy']*100:.2f}%)")
                # 여기서 1등 모델만 다시 빌드해서 저장하는 로직을 추가할 수도 있습니다.
                
        else:
            print(f"\n[Error] 로드된 모델이 {len(available_models)}개입니다. 최소 3개가 필요합니다.")

📂 Loading Data...

📂 Loading Base Models...
  > Loaded & Wrapped: mlp
  > Loaded & Wrapped: rf
  > Loaded & Wrapped: knn
  > Loaded & Wrapped: gb

🚀 Starting Combinations Experiment (Stacking 6 Meta & Voting)
   Base Models Available: 4 (['mlp', 'rf', 'knn', 'gb'])

[3 Models Combo] mlp+rf+knn
Acc: 99.50% (7.8s)
Acc: 49.70% (1.3s)
Acc: 99.45% (5.7s)
Acc: 99.44%
Acc: 99.38%

[3 Models Combo] mlp+rf+gb
Acc: 99.39% (7.2s)
Acc: 49.74% (1.6s)
Acc: 99.24% (4.7s)
Acc: 99.16%
Acc: 99.07%

[3 Models Combo] mlp+knn+gb
Acc: 99.40% (4.4s)
Acc: 49.70% (1.2s)
Acc: 99.40% (6.4s)
Acc: 99.27%
Acc: 99.18%

[3 Models Combo] rf+knn+gb
Acc: 99.33% (5.0s)
Acc: 49.78% (1.2s)
Acc: 99.30% (5.0s)
Acc: 99.19%
Acc: 99.12%

[4 Models Combo] mlp+rf+knn+gb
Acc: 99.48% (6.9s)
Acc: 49.70% (1.7s)
Acc: 99.45% (6.2s)
Acc: 99.38%
Acc: 99.25%

🏆 Top 20 Best Ensemble Results
             Type    Base Models Meta Model  Accuracy       Time
0        Stacking     mlp+rf+knn   Logistic  0.995012   7.818375
20       Stacking  ml